In [1]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors
# from adjustText import adjust_text
import pandas as pd
from scipy.stats import pearsonr, spearmanr, kendalltau
import numpy as np
import seaborn as sns
from matplotlib.patches import Patch
import sys

sys.path.insert(1,'../4. CompareNormalisation')
from Functions import *
import MetricMapping

# Access the mappings:
type_mapping = MetricMapping.type_mapping
name_mapping = MetricMapping.name_mapping

label_resolutions=["5 minute", "10 minute", "30 minute", "60 minute"]
resolution_index_res = {'5m': 0 , '10m': 1, '30m': 2, '60m': 3}

In [2]:
datadir = '/scratch/hydro4/users/kv25483/MetricEvaluation/Data/IntermediateFiles/'
figdir = '/scratch/hydro4/users/kv25483/MetricEvaluation/Figures/'

### Read in data and remove the log/yj endings

In [3]:
Justraw = pd.read_csv(datadir+"Justraw.csv")

#### Delineate categorical and continuous metrics

In [4]:
metrics = list(name_mapping.keys())
categorical_metrics = ['3rd_ARR',  '3rd_rcg',  '3rd_w_peak', '4th_w_peak', '5th_w_peak', 'third_ppr', '3rd_w_most', 
                       '4th_w_most', '5th_w_most']
continuous_metrics = [metric for metric in list(type_mapping.keys()) if metric not in categorical_metrics]

# categorical_metrics = categorical_metrics = [col + '_raw' for col in categorical_metrics]
# continuous_metrics = continuous_metrics = [col + '_raw' for col in continuous_metrics]

In [5]:
summary_df = compute_metric_sensitivity_by_resolution(df=transformed_minmax_scaled,
    continuous_metrics=continuous_metrics,
    categorical_metrics=categorical_metrics,
    resolutions=["10m", "30m", "60m"])

summary_df["type2"] = summary_df["metric"].map(type_mapping)

NameError: name 'transformed_minmax_scaled' is not defined

In [ ]:
# Split intermittency and other types
df_continuous = summary_df[summary_df["type"] != "categorical"]
unique_metrics_continuous = df_continuous["metric"].unique()

df_categorical = summary_df[summary_df["type"] == "categorical"]
unique_metrics_categorical = df_categorical["metric"].unique()

## Plot

In [ ]:
n_cols_main = 6
n_rows_main = -(-len(unique_metrics_continuous) // n_cols_main)

fig_main, axs_main = plt.subplots(ncols=n_cols_main, nrows=n_rows_main,
                                  figsize=(4.2 * n_cols_main, 3.5 * n_rows_main),
                                  sharex=True, sharey=True)
axs_main = axs_main.flatten()

# Track max/min values for autoscaled limits
x_min, x_max, y_min, y_max = float('inf'), float('-inf'), float('inf'), float('-inf')


for i, this_metric in enumerate(unique_metrics_continuous):
    ax = axs_main[i]
    metric_name_for_plot = name_mapping[this_metric]
    metric_data = df_continuous[df_continuous["metric"] == this_metric]
    scatter_without_labels(ax, metric_data, metric_name_for_plot, type_color_map_1, resolution_index_res)

    # Update global x/y limits
    x_vals = metric_data["rank_corr"]
    y_vals = metric_data["val_diff"]
    if not x_vals.empty and not y_vals.empty:
        x_min = min(x_min, x_vals.min())
        x_max = max(x_max, x_vals.max())
        y_min = min(y_min, y_vals.min())
        y_max = max(y_max, y_vals.max())
        
        # Round outer limits to nearest multiple of 5 or 10
        x_min_rounded = np.floor(x_min / 5) * 1
        x_max_rounded = np.ceil(x_max / 5) * 1
        y_min_rounded = np.floor(y_min / 5) * 5
        y_max_rounded = np.ceil(y_max / 5) * 5

        # Set tick positions every 5 or 10 units
        xticks = np.arange(x_min_rounded, x_max_rounded +0.25, 0.25)
        yticks = np.arange(y_min_rounded, y_max_rounded + 1, 25)

        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)        
        ax.set_xticks(xticks)
        ax.set_yticks(yticks)
        ax.grid(True)

    for i, ax in enumerate(axs_main):
        if i % n_cols_main == 0:  # First column in each row
            ax.set_ylabel("sMAPE from 5m", fontsize=23)
        else:
            ax.set_ylabel('')    
    for i, ax in enumerate(axs_main):
        row = i // n_cols_main
        if row == n_rows_main - 1:  # Last row
            ax.set_xlabel("Spearman’s ρ", fontsize=23)
        else:
            ax.set_xlabel('')

for ax in axs_main[len(unique_metrics_continuous):]:
    ax.axis('off')

fig_main.suptitle("(a). Continuous metrics", fontsize=40)
plt.tight_layout(rect=[0, 0, 1, 0.96])
# fig_main.savefig(figdir+ "CompareResolutions/Scatter_Continuous.png", dpi=300, facecolor='white')

### Categorical

In [ ]:
def scatter_without_labels(ax, data, this_metric, type_color_map, resolution_index_res):
    for _, row in data.iterrows():
        metric_type = row["type2"]
        resolution = row["resolution"]
        color = type_color_map[metric_type][resolution_index_res[resolution]]
        ax.scatter(row["rank_corr"], row["val_diff"], color=color, edgecolor='black',
                   marker='o', s=550, alpha=1)
    ax.set_title(this_metric, fontsize=25, fontstyle="italic")
    ax.grid(True)
    ax.tick_params(axis='both', which='major', labelsize=9)
    ax.set_xlabel("Spearman’s ρ", fontsize=15)
    ax.set_ylabel("MAD from 5m", fontsize=15)
    
    # Highlight box
    x_min, x_max = 0.9, 1.001
    y_min, y_max = 0, 0.2
    
    ax.tick_params(axis='both', which='major', labelsize=15)

In [ ]:
resolutions = ["10m", "30m", "60m"]

# Plot setup
n_cols = 6
n_rows = -(-len(unique_metrics_categorical) // n_cols)
fig, axs = plt.subplots(ncols=n_cols, nrows=n_rows, figsize=(4.2 * n_cols, 4 * n_rows), sharex=True, sharey=True)
axs = axs.flatten()

# Plot each metric
for i, this_metric in enumerate(unique_metrics_categorical):
    ax = axs[i]
    metric_name_for_plot = name_mapping[this_metric]
    metric_data = df_categorical[df_categorical["metric"] == this_metric]
    scatter_without_labels(ax, metric_data, metric_name_for_plot, type_color_map_1, resolution_index_res)

# Hide unused axes
for ax in axs[len(unique_metrics_categorical):]:
    ax.axis('off')

for i, ax in enumerate(axs):
    row = i // n_cols
    if row == n_rows - 1:  # Last row
        ax.set_xlabel("Kendall’s τ", fontsize=23)
    else:
        ax.set_xlabel('')   
        
for i, ax in enumerate(axs):
    if i % n_cols == 0:  # First column in each row
        ax.set_ylabel("% diff. from 5m", fontsize=23)
    else:
        ax.set_ylabel('')     


# Save
fig.suptitle("(b). Categorical metrics", fontsize=40)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()
fig.savefig(figdir+ "CompareResolutions/Scatter_Categorical.png", dpi=300, facecolor='white')

### Create legend

In [ ]:
legend_types = list(type_color_map_2.keys())
labels = ['10 minute', '30 minute', '60 minute']

fig_leg, axs_leg = plt.subplots(ncols=len(legend_types),figsize=(5 * len(legend_types), 2))

if len(legend_types) == 1:
    axs_leg = [axs_leg]

for ax, metric_type in zip(axs_leg, legend_types):

    ax.axis("off")

    colors = type_color_map_2[metric_type]

    patches = [Patch(facecolor=colors[j], label=labels[j], alpha=0.8)
        for j in range(len(resolutions))]
    
    # Handle long titles
    title = metric_type.replace(" ", "\n", 1) if len(metric_type) > 20 else metric_type
    if "\n" not in title:
        title = title + "\n "  # blank second line for spacing

    # Legend without title
    ax.legend(handles=patches, title='',loc="center",frameon=False, fontsize=20, handlelength=6, handleheight=4)

    # Add text above the color boxes
    ax.text(0, 1.5, title, fontsize=25, ha='left', va='bottom', transform=ax.transAxes)

# --- manually adjust top margin so text is not cut off ---
fig_leg.subplots_adjust(top=0.85, left=0.05, right=0.95)  # adjust top, left, right margins
plt.show()
# fig_leg.savefig(figdir+"CompareResolutions/Scatter_Legend.png", dpi=300, facecolor='white', bbox_inches='tight')

### Join the two images

In [ ]:
from PIL import Image
import glob

# Collect all PNGs (adjust pattern/order as needed)
files = sorted(glob.glob(figdir + "CompareResolutions/Scatter_Continuous.png")) + \
        sorted(glob.glob(figdir+"CompareResolutions/Scatter_Categorical.png")) + \
        sorted(glob.glob(figdir+"CompareResolutions/Scatter_Legend.png"))

if not files:
    raise ValueError("No images found – check path or extension")

# Open all images
imgs = [Image.open(f).convert("RGB") for f in files]

# All same width? If not, resize to match the widest
width = max(img.width for img in imgs)
resized = [img if img.width == width else img.resize((width, int(img.height * width/img.width)))
           for img in imgs]


# Add extra spacing after the 2nd image
spacing_after_second = 210  

total_height = sum(img.height for img in resized)+ spacing_after_second + spacing_after_second
canvas = Image.new("RGB", (width, total_height), (255, 255, 255))

y = 0
for i, img in enumerate(resized):
    canvas.paste(img, (0, y))
    y += img.height
    # add spacing after 2nd image
    if i in [0,1]:
        y += spacing_after_second    
    
canvas.save(figdir + "CompareResolutions/Scatter_Combined.png", dpi=(300, 300))